# 08 - Topic-Level Watermark Detection (uniform static greenlists)

Detector for the **first layer** (public, topic-based watermark) generated by
`topic_wise_watermarking.ipynb` / `src/watermark/first_layer.py`: **static uniform**
greenlists, one CSV per topic under `data/greenlist/<topic>.csv` with columns

    token_id, token, similarity, type ∈ {content, connector, residual}

Connectors *and* below-threshold residuals were round-robined across topics, so
every list covers the whole vocabulary (γ = |G_t| / vocab ≈ 0.08–0.22).

## The scheme being detected
1. `extract_topic(prompt)` — mean-pool the prompt's token embeddings in OPT's own
   embedding space, cosine-compare against one vector per predetermined topic, take argmax.
2. Boost that topic's **fixed greenlist** logits by `delta` at *every* generation step
   (`TopicBoostProcessor`, split `"all"`).

## Detection as a hypothesis test
Because the greenlist is fixed per topic (not context-dependent like KGW), every
scored token is an independent Bernoulli trial:

- **H0** (text is NOT watermarked): each token lands in topic `t`'s greenlist with probability  
  `gamma_t = |G_t| / vocab_size`
- **H1** (watermarked with delta > 0): green tokens are over-represented

Given `s` green hits among `n` scored tokens:

```
z = (s/n - gamma) / sqrt(gamma*(1-gamma)/n)
p_value = P(Binom(n, gamma) >= s)   # survival function, upper tail
```

Verdict = watermarked iff `z >= Z_THRESHOLD` AND `p_value < P_VALUE_THRESHOLD`.

> Same statistical core as `src/detection/kgw_detection.py`; the difference is where the
> green set comes from: looked up from the **public per-topic CSVs** via the re-inferred
> topic instead of derived from a secret key + token history. Greenlist loading, topic
> geometry and the boost processor are imported straight from
> `src/watermark/first_layer.py` (`load_uniform_greenlists`, `build_topic_matrix`,
> `TopicBoostProcessor`) so detection and generation can never drift apart.

## Setup — Colab VM only

Clone the repo, enter it, install runtime deps, log in to HF.
(Nothing here is meant to run on a local machine.)

In [3]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git
%cd /content/Dual_watermarking_Scheme

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 216, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 216 (delta 30), reused 40 (delta 11), pack-reused 151 (from 1)
Receiving objects: 100% (216/216), 43.48 MiB | 23.83 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/Dual_watermarking_Scheme


In [4]:
!pip install transformers scipy pandas

In [1]:
import os
import torch
from huggingface_hub import login
login()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


device: cuda


In [5]:
import math
import pandas as pd


GREENLIST_DIR     = "data/greenlist"  # per-topic CSVs from 05_greenlist_construction.ipynb
MODEL_NAME        = "facebook/opt-2.7b"
SPLIT             = "all"             # MUST match the split boosted at generation time
Z_THRESHOLD       = 4.0               # z must exceed this (KGW-style tau)
P_VALUE_THRESHOLD = 0.05              # ... and the upper-tail binomial p must be below this

DELTA_SWEEP       = [2.0]
MAX_NEW_TOKENS    = 200               # pipeline's dual-layer operating
                                      # length; z grows ~sqrt(n), short texts lack power

assert os.path.isdir(GREENLIST_DIR), f"{GREENLIST_DIR}/ not found - clone the repo first"
print("greenlist dir :", GREENLIST_DIR)
print("split         :", SPLIT)

greenlist dir : data/greenlist
split         : all


## Step 1 - Reusable pieces from `src/watermark/first_layer.py`

Identical source of truth as the generation side:

- `load_uniform_greenlists(dir)` -> `{topic: {"all": [...], "content": [...]}}`
- `build_topic_matrix(model, tokenizer, topics)` -> unit-norm topic vectors
- `TopicBoostProcessor(ids, delta)` — replayed in the live end-to-end sweep

The greenlist is **public and fixed per topic**, so detection needs no secret key -
only the CSVs, the vocab size, and the ability to re-infer the topic. `SPLIT` must
match the one boosted during generation (`"all"` = paper-style full coverage,
`"content"` = strict content words only).

In [6]:
import sys
sys.path.insert(0, "..")   # running from notebooks/
sys.path.insert(0, ".")    # running from repo root after %cd

from src.watermark.first_layer import (
    TopicBoostProcessor,
    build_topic_matrix,
    load_uniform_greenlists,
)

greenlists = load_uniform_greenlists(GREENLIST_DIR)
TOPICS = sorted(greenlists)

# freeze the chosen split into lookup sets (detection never touches "all"/"content" again)
green_sets = {t: frozenset(v[SPLIT]) for t, v in greenlists.items()}
print("topics:", TOPICS)
print({t: len(s) for t, s in green_sets.items()})

topics: ['entertainment', 'finance', 'history', 'medicine', 'politics', 'science', 'sports', 'technology']
{'entertainment': 4244, 'finance': 4236, 'history': 5803, 'medicine': 4230, 'politics': 7701, 'science': 6092, 'sports': 6710, 'technology': 11240}


## Step 2 - Model, embeddings, topic matrix

Same geometry that built the greenlists (`04_cosine_similarity.ipynb`,
`05_greenlist_construction.ipynb`) and drove generation
(`topic_wise_watermarking.ipynb`):

- unit-normalize OPT's input embedding table -> a dot product between rows **is** a cosine
- topic vector = embedding of the first subtoken of `" " + topic` (e.g. `Ġtechnology`),
  built here via `first_layer.build_topic_matrix`

Detection must use the **same** embedding space as generation, otherwise the inferred
topic can drift away from the one used to pick the greenlist.

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device).eval()

# NOTE: OPT quirk -- len(tokenizer) = 50265 usable ids, but the embedding
# matrix / config.vocab_size = 50272 rows (7 reserved slots, never emitted).
# Greenlists were built over the tokenizer id-space (opt27_vocab.csv), so
# len(tokenizer) is the correct denominator for gamma (same as generation).
emb_rows = model.get_input_embeddings().weight.shape[0]
assert emb_rows >= len(tokenizer)
VOCAB_SIZE = len(tokenizer)
print(f"usable ids: {len(tokenizer)} | embedding rows: {emb_rows}")
print("vocab size (gamma denominator):", VOCAB_SIZE)

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.30GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

usable ids: 50265 | embedding rows: 50272
vocab size (gamma denominator): 50265


In [8]:
emb = model.get_input_embeddings().weight.detach().float()
normed_embeddings = emb / emb.norm(dim=1, keepdim=True)

topic_matrix = build_topic_matrix(model, tokenizer, TOPICS)   # (K, d), unit-norm rows
for t in TOPICS:
    tid = tokenizer.encode(" " + t, add_special_tokens=False)[0]
    print(f"{t:14} -> {tokenizer.decode([tid])!r} (id {tid})")
print("topic_matrix:", tuple(topic_matrix.shape))

entertainment  -> ' entertainment' (id 4000)
finance        -> ' finance' (id 2879)
history        -> ' history' (id 750)
medicine       -> ' medicine' (id 6150)
politics       -> ' politics' (id 2302)
science        -> ' science' (id 2866)
sports         -> ' sports' (id 1612)
technology     -> ' technology' (id 806)
topic_matrix: (8, 2560)


## Step 3 - Re-infer the topic from the text under test

At detection time we only have the text (the prompt may be unknown), so we run the
**same** `extract_topic` logic on the generated text itself. The boosted tokens pull
their own topic to the top, which is exactly what routes us to the correct greenlist.

In [17]:
@torch.no_grad()
def rank_topics(text):
    """Cosine of mean-pooled text embedding vs each topic vector.

    Mirrors first_layer.TopicWiseWatermarking.extract_topic (generation side):
    mean-pool token embeddings of `text`, normalize, dot with the topic matrix.
    """
    ids = tokenizer.encode(text, add_special_tokens=False)
    if not ids:
        return []
    vec = normed_embeddings[ids].mean(dim=0)
    vec = vec / vec.norm()
    scores = (topic_matrix @ vec).tolist()          # (K,) cosine per topic
    return sorted(zip(TOPICS, scores), key=lambda x: -x[1])


def extract_topic(text, verbose=False):
    ranked = rank_topics(text)
    if verbose:
        for name, s in ranked:
            bar = "#" * max(0, int((s + 1) * 20))   # cosine [-1, 1] -> bar
            print(f"  {name:14} {s:+.4f} {bar}")
    return (ranked[0][0], ranked) if ranked else (None, [])

In [18]:
# quick feel for the geometry on a few off-the-shelf texts
for p in [
    ("The government passed a controversial new surveillance law just before the "
     "national election, and opposition parties immediately challenged it in court"),
    ("The cricket team chased down 280 runs in the final match and won the trophy "
     "on the very last ball of a thrilling tournament"),
]:
    print(f"\n{p!r}")
    best, _ = extract_topic(p, verbose=True)
    print("->", best)



'The government passed a controversial new surveillance law just before the national election, and opposition parties immediately challenged it in court'
  history        +0.2637 #########################
  technology     +0.2533 #########################
  sports         +0.2214 ########################
  science        +0.2039 ########################
  politics       +0.1674 #######################
  entertainment  +0.1294 ######################
  medicine       +0.1016 ######################
  finance        +0.0987 #####################
-> history

'The cricket team chased down 280 runs in the final match and won the trophy on the very last ball of a thrilling tournament'
  sports         +0.2649 #########################
  history        +0.2545 #########################
  technology     +0.2186 ########################
  science        +0.1813 #######################
  entertainment  +0.1175 ######################
  politics       +0.1074 ######################
  medicine      

## Step 4 - The detection statistic

For topic `t` with greenlist `G_t` (`gamma_t = |G_t| / vocab_size`) and a text of `n`
scored tokens containing `s` green hits:

```
green_rate   = s / n
expected     = n * gamma_t
std          = sqrt(n * gamma_t * (1 - gamma_t))
z            = (s - expected) / std          ~ N(0, 1) under H0
p_value      = binomtest(s, n, gamma_t, alternative='greater').pvalue
```

Notes
- Unlike KGW there is **no context window / prev-token hash**: the greenlist is fixed,
  so *every* token position is scoreable (n is larger -> more power).
- **Length is the main power knob**: under H1, z grows roughly like
  `(green_rate - gamma) * sqrt(n / (gamma*(1-gamma)))`, so a modest per-token lift
  (delta = 2) cannot clear z >= 4 on a sentence's worth of tokens (~20-40). Use
  150+ scored tokens before trusting a negative result.
- With the **uniform** lists every topic now sits at gamma ≈ 0.08–0.22 (the legacy
  single-CSV lists ranged 0.002–0.14), so one decision rule calibrates sensibly across
  topics. Still always use the topic's *own* gamma, never a global rate.
- `p_value` is exact binomial; z is its normal approximation. We report both because
  z is interpretable as "how many standard deviations above chance", while p is what
  the decision rule thresholds.

In [19]:
from scipy.stats import binomtest

for t in TOPICS:
    gamma_t = len(green_sets[t]) / VOCAB_SIZE
    print(f"{t:14} |G_t|={len(green_sets[t]):>6}  gamma_t={gamma_t:.4f}")


def detection_stats(token_ids, topic):
    """Count green hits against the topic's own gamma -> z-score + exact binomial p-value."""
    gset = green_sets[topic]
    gamma = len(gset) / VOCAB_SIZE

    hits = sum(1 for tid in token_ids if tid in gset)
    n = len(token_ids)

    if n == 0 or gamma == 0:
        return {"hits": hits, "num_positions": n, "gamma": gamma,
                "green_rate": 0.0, "z_score": 0.0, "p_value": 1.0}

    expected = n * gamma
    std = math.sqrt(n * gamma * (1 - gamma))
    z = (hits - expected) / std
    p_value = binomtest(k=hits, n=n, p=gamma, alternative="greater").pvalue

    return {"hits": hits, "num_positions": n, "gamma": round(gamma, 5),
            "green_rate": round(hits / n, 4),
            "z_score": round(z, 4), "p_value": p_value}

entertainment  |G_t|=  4244  gamma_t=0.0844
finance        |G_t|=  4236  gamma_t=0.0843
history        |G_t|=  5803  gamma_t=0.1154
medicine       |G_t|=  4230  gamma_t=0.0842
politics       |G_t|=  7701  gamma_t=0.1532
science        |G_t|=  6092  gamma_t=0.1212
sports         |G_t|=  6710  gamma_t=0.1335
technology     |G_t|= 11240  gamma_t=0.2236


In [20]:
def detect_topic_watermark(text, z_threshold=Z_THRESHOLD,
                           p_value_threshold=P_VALUE_THRESHOLD, verbose=False):
    """Full first-layer detection: infer topic -> green hits -> z & p decision.

    Returns a dict in the same spirit as kgw_detection.detect_private_watermark,
    plus the topic fields (which topic the text was routed to and how confidently):
      topic, topic_score, ownership_score (= s/n), match_count, num_positions,
      gamma, z_score, p_value, confirmed
    """
    best_topic, ranked = extract_topic(text)
    if best_topic is None:
        return {"topic": None, "topic_score": None, "ownership_score": 0.0,
                "match_count": 0, "num_positions": 0, "gamma": 0.0,
                "z_score": 0.0, "p_value": 1.0, "confirmed": False}

    token_ids = tokenizer.encode(text, add_special_tokens=False)
    stats = detection_stats(token_ids, best_topic)

    confirmed = bool(stats["z_score"] >= z_threshold
                     and stats["p_value"] < p_value_threshold)

    result = {
        "topic": best_topic,
        "topic_score": round(ranked[0][1], 4),
        "ownership_score": stats["green_rate"],
        "match_count": stats["hits"],
        "num_positions": stats["num_positions"],
        "gamma": stats["gamma"],
        "z_score": stats["z_score"],
        "p_value": stats["p_value"],
        "confirmed": confirmed,
    }

    if verbose:
        print(f"inferred topic : {best_topic} (cosine {result['topic_score']:+.4f})")
        print(f"green hits     : {stats['hits']}/{stats['num_positions']}"
              f"  (expected {stats['gamma'] * stats['num_positions']:.1f}, gamma={stats['gamma']})")
        print(f"z-score        : {stats['z_score']:+.2f}  (threshold {z_threshold})")
        print(f"p-value        : {stats['p_value']:.2e}  (threshold {p_value_threshold})")
        print(f"watermarked?   : {'YES' if confirmed else 'no'}")

    return result

### Smoke test on hand-written text

Human/other-LLM text should score near `z ~ 0`, `p ~ 0.5` and NOT be flagged.
The two texts below also probe that topic routing lands on a *sensible* greenlist.

In [21]:
# unwatermarked human-ish text -> expect z near 0, p large, confirmed=False
for t in [
    ("The quick brown fox jumps over the lazy dog near the river bank, then trots "
     "off into the tall grass behind the old barn where the hens sleep until sunrise."),
    ("She packed her bags late at night, booked a flight for the following morning "
     "and left without saying goodbye, leaving nothing behind except a short "
     "handwritten note on the kitchen table."),
    ("My grandmother used to wake up before dawn every single day, walk to the small "
     "market two streets over, and argue cheerfully with the vegetable seller about "
     "the price of tomatoes."),
]:
    print(f"\n{t!r}")
    _ = detect_topic_watermark(t, verbose=True)



'The quick brown fox jumps over the lazy dog near the river bank, then trots off into the tall grass behind the old barn where the hens sleep until sunrise.'
inferred topic : history (cosine +0.2495)
green hits     : 6/34  (expected 3.9, gamma=0.11545)
z-score        : +1.11  (threshold 4.0)
p-value        : 1.93e-01  (threshold 0.05)
watermarked?   : no

'She packed her bags late at night, booked a flight for the following morning and left without saying goodbye, leaving nothing behind except a short handwritten note on the kitchen table.'
inferred topic : history (cosine +0.2627)
green hits     : 2/34  (expected 3.9, gamma=0.11545)
z-score        : -1.03  (threshold 4.0)
p-value        : 9.16e-01  (threshold 0.05)
watermarked?   : no

'My grandmother used to wake up before dawn every single day, walk to the small market two streets over, and argue cheerfully with the vegetable seller about the price of tomatoes.'
inferred topic : history (cosine +0.2576)
green hits     : 6/35  (expe

## Step 5 - Sanity check on saved generation outputs

`data/example/topic_wise_watermarked_example.csv` holds `plain_output` vs
`watermarked_output` pairs produced by the generation notebook. The detector should
separate them: high z / tiny p for watermarked, near-null stats for plain.

> The pairs must come from the **new** uniform-list pipeline
> (`topic_wise_watermarking.ipynb`). If the CSV still predates it, expect weak
> separation - regenerate the example first.

In [22]:
EXAMPLES_CSV = "data/example/topic_wise_watermarked_example.csv"
examples = pd.read_csv(EXAMPLES_CSV)
print(examples[["prompt", "topic"]])
examples.head()

                                              prompt       topic
0  The government passed a new law before the ele...     history
1               The cricket team won the final match      sports
2                 The new AI chip runs twice as fast  technology


,id,prompt,topic,topic_score,plain_output,watermarked_output
0,0,The government passed a new law before the ele...,history,0.2512,The government passed a new law before the ele...,The government passed a new law before the ele...
1,1,The cricket team won the final match,sports,0.2634,The cricket team won the final match of the se...,The cricket team won the final match of the To...
2,2,The new AI chip runs twice as fast,technology,0.2389,The new AI chip runs twice as fast as a standa...,The new AI chip runs twice as fast as today's ...


In [23]:
print(examples.iloc[1]['plain_output'])
print(examples.iloc[1]['watermarked_output'])

The cricket team won the final match of the season, against the Kandy Tuskers by the score of 9 runs. The match was close with a total of 169 runs on board. Chasing 171, the Tuskers lost early wickets but Shaiman Anwar and Shahadat
The cricket team won the final match of the Tournament
Society The cricket team won the final match of the Tournament SportsSociety The cricket team won the final match of the Tournament Source: B92 Thursday, February 11, 2021 | 17:03 Tweet Share Sportskphoto
The


In [24]:
rows = []
for _, r in examples.iterrows():
    for col, label in (("plain_output", "plain"), ("watermarked_output", "watermarked")):
        res = detect_topic_watermark(r[col])
        rows.append({
            "prompt": r["prompt"],
            "kind": label,
            "gen_topic": r["topic"],          # topic assigned at generation time
            "detected_topic": res["topic"],   # topic re-inferred by detector
            "hits": res["match_count"],
            "n": res["num_positions"],
            "gamma": res["gamma"],
            "green_rate": res["ownership_score"],
            "z_score": res["z_score"],
            "p_value": res["p_value"],
            "confirmed": res["confirmed"],
        })
sanity_df = pd.DataFrame(rows)
pd.set_option("display.width", 160)
sanity_df

,prompt,kind,gen_topic,detected_topic,hits,n,gamma,green_rate,z_score,p_value,confirmed
0,The government passed a new law before the ele...,plain,history,history,3,59,0.11545,0.0508,-1.5528,0.972792,False
1,The government passed a new law before the ele...,watermarked,history,history,9,59,0.11545,0.1525,0.8916,0.237022,False
2,The cricket team won the final match,plain,sports,history,3,57,0.11545,0.0526,-1.4841,0.967266,False
3,The cricket team won the final match,watermarked,sports,sports,14,57,0.13349,0.2456,2.4889,0.015928,False
4,The new AI chip runs twice as fast,plain,technology,technology,6,58,0.22361,0.1034,-2.1964,0.994569,False
5,The new AI chip runs twice as fast,watermarked,technology,technology,24,58,0.22361,0.4138,3.4760,0.000922,False


In [25]:
# topic routing check: did the detector recover the generation-time topic?
wm_rows = sanity_df[sanity_df.kind == "watermarked"]
routing_ok = wm_rows["gen_topic"] == wm_rows["detected_topic"]
print("topic recovered on watermarked texts:", f"{routing_ok.mean():.0%}")
print("\nmean z: plain = %.2f | watermarked = %.2f" % (
    sanity_df[sanity_df.kind == "plain"].z_score.mean(),
    wm_rows.z_score.mean()))

topic recovered on watermarked texts: 100%

mean z: plain = -1.74 | watermarked = 2.29


## Step 6 - Live end-to-end: generate -> detect

Fresh generations (not the saved CSV) using `first_layer.TopicBoostProcessor`,
sweeping `delta`. Expected behaviour:

- `delta = 0` behaves like the plain model -> z near 0
- larger delta pushes more tokens into the greenlist -> z climbs
- detection flips to `confirmed` once z crosses the threshold and p collapses
Length is the power bottleneck: with `MAX_NEW_TOKENS = 200` there are enough scored
tokens for delta = 2 to register, whereas a 60-token run often stalls around
z ~ 2-3. Two things eat into n: sampling occasionally produces fewer usable
positions, and the **plain prompt prefix is scored too but is never watermarked**,
so it dilutes the green rate toward H0 - keep the continuation much longer than
the prompt.


In [29]:
PROMPT = ("The government passed a controversial new law just before the election, "
          "and the opposition immediately challenged it in court, arguing that the "
          "rushed midnight vote in parliament ignored decades of constitutional "
          "precedent and stripped citizens of long-standing protections.")
n_prompt = len(tokenizer.encode(PROMPT, add_special_tokens=False))
print(f"prompt tokens: {n_prompt}")
topic, _ = extract_topic(PROMPT)
print("assigned topic:", topic)

sweep_rows = []
for d in DELTA_SWEEP:
    inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
    processors = ([TopicBoostProcessor(greenlists[topic][SPLIT], delta=d)]
                  if d > 0 else None)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                             temperature=1.0, top_p=0.9,
                             logits_processor=processors)
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    det = detect_topic_watermark(text)
    sweep_rows.append({"delta": d, "detected_topic": det["topic"],
                       "hits": det["match_count"], "n": det["num_positions"],
                       "green_rate": det["ownership_score"], "gamma": det["gamma"],
                       "z_score": det["z_score"], "p_value": det["p_value"],
                       "confirmed": det["confirmed"]})
    print(f"delta={d:>4}  z={det['z_score']:+7.2f}  p={det['p_value']:.3e}"
          f"  confirmed={det['confirmed']}\n  {text[:110]}...")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df

prompt tokens: 43
assigned topic: history
delta= 2.0  z= +10.03  p=1.532e-17  confirmed=True
  The government passed a controversial new law just before the election, and the opposition immediately challen...


,delta,detected_topic,hits,n,green_rate,gamma,z_score,p_value,confirmed
0,2.0,history,78,243,0.321,0.11545,10.0264,1.531566e-17,True


## Step 9 - Persist results

## Export — `src/detection/topic_detection.py`

Freezes the verified notebook logic into the first-layer counterpart of
`kgw_detection.py`, same shape (module constants + plain `detect_*` functions):

- `prepare(model, tokenizer, greenlist_dir, split)` — builds the shared state once
  (per-topic green sets, unit-norm embeddings, topic matrix) via
  `first_layer.load_uniform_greenlists` / `build_topic_matrix`
- `_extract_topic` — same routing math as `TopicWiseWatermarking.extract_topic`
- `detect_topic_watermark(text, tokenizer, state, vocab_size, ...)` — green hits,
  z-score + exact binomial p-value verdict dict (topic fields included)
- `detect_dataframe(df, tokenizer, state, text_column="text", ...)` — original
  columns + detection columns, mirroring `kgw_detection.detect_dataframe`

Path assumes the notebook runs from the repo root (after `%cd` on Colab).


In [ ]:
%%writefile topic_detection.py
'''First-layer (public, topic-based) watermark detection.
re-infers the topic
from the text via cosine similarity in OPT's embedding space, looks up that
topic's fixed greenlist (static uniform CSVs under data/greenlist/<topic>.csv),
counts green-token hits and tests them against the H0 rate gamma = |G|/vocab_size
with a z-score plus an exact binomial upper-tail p-value.
'''
import math

import pandas as pd
import torch
from scipy.stats import binomtest

Z_THRESHOLD = 4.0
P_VALUE_THRESHOLD = 0.05

from src.watermark.first_layer import build_topic_matrix, load_uniform_greenlists


def prepare(
    model,
    tokenizer,
    greenlist_dir="data/greenlist",
    split="all",
):
    """
        builds the shared detection state once: per-topic green sets for the given
        split plus the unit-norm embedding table and topic matrix used for routing.
        MUST use the same model/tokenizer/dir/split as generation so routing and
        gamma match. returns dict: topics, green_sets, normed_embeddings, topic_matrix
    """

    greenlists = load_uniform_greenlists(greenlist_dir)
    topics = sorted(greenlists)
    green_sets = {t: frozenset(v[split]) for t, v in greenlists.items()}

    emb = model.get_input_embeddings().weight.detach().float()
    normed_embeddings = emb / emb.norm(dim=1, keepdim=True)

    return {
        "topics": topics,
        "green_sets": green_sets,
        "normed_embeddings": normed_embeddings,
        "topic_matrix": build_topic_matrix(model, tokenizer, topics),
    }


def _extract_topic(
    text,
    tokenizer,
    state,
):
    """
        re-infers the topic of `text`; mirrors first_layer.TopicWiseWatermarking.
        extract_topic so detection routes identically to generation.
        returns (best_topic, best_cosine), or (None, 0.0) for empty text.
    """

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )
    if not token_ids:
        return None, 0.0

    vec = state["normed_embeddings"][token_ids].mean(dim=0)
    vec = vec / vec.norm()

    scores = (state["topic_matrix"] @ vec).tolist()
    ranked = sorted(zip(state["topics"], scores), key=lambda x: -x[1])

    return ranked[0][0], ranked[0][1]


def detect_topic_watermark(
    text,
    tokenizer,
    state,
    vocab_size,
    z_threshold=Z_THRESHOLD,
    p_value_threshold=P_VALUE_THRESHOLD,
):
    """
        returns dictionary containing: topic, topic_score, ownership_score,
        match_count, num_positions, gamma, z_score, p_value, confirmed
    """

    topic, topic_score = _extract_topic(text, tokenizer, state)

    if topic is None:
        return {"topic": None, "topic_score": None, "ownership_score": 0.0,
                "match_count": 0, "num_positions": 0, "gamma": 0.0,
                "z_score": 0.0, "p_value": 1.0, "confirmed": False}

    green_set = state["green_sets"][topic]
    gamma = len(green_set) / vocab_size

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    matches = sum(1 for tid in token_ids if tid in green_set)
    total_positions = len(token_ids)

    ownership_score = (
        matches / total_positions
        if total_positions > 0
        else 0.0
    )

    if total_positions > 0 and gamma > 0:

        z_score = (
            (matches - total_positions * gamma)
            / math.sqrt(total_positions * gamma * (1 - gamma))
        )

        p_value = binomtest(
            k=matches,
            n=total_positions,
            p=gamma,
            alternative="greater",
        ).pvalue

    else:
        z_score = 0.0
        p_value = 1.0

    confirmed = (
        z_score >= z_threshold
        and p_value < p_value_threshold
    )

    return {
        "topic": topic,
        "topic_score": round(topic_score, 4),
        "ownership_score": round(ownership_score, 4),
        "match_count": matches,
        "num_positions": total_positions,
        "gamma": round(gamma, 5),
        "z_score": round(z_score, 4),
        "p_value": p_value,
        "confirmed": bool(confirmed),
    }


def detect_dataframe(
    df,
    tokenizer,
    state,
    text_column="text",
    vocab_size=None,
    z_threshold=Z_THRESHOLD,
    p_value_threshold=P_VALUE_THRESHOLD,
):
    """
        this function will return the df containing the original data plus detection results.
    """

    if vocab_size is None:
        vocab_size = len(tokenizer)

    detection_results = []

    for _, row in df.iterrows():

        result = detect_topic_watermark(
            text=row[text_column],
            tokenizer=tokenizer,
            state=state,
            vocab_size=vocab_size,
            z_threshold=z_threshold,
            p_value_threshold=p_value_threshold,
        )

        detection_results.append(result)

    detection_df = pd.DataFrame(detection_results)

    return pd.concat(
        [df.reset_index(drop=True),
         detection_df.reset_index(drop=True)],
        axis=1,
    )


Writing topic_detection.py


In [32]:
import importlib
import topic_detection

importlib.reload(topic_detection)

state = topic_detection.prepare(model, tokenizer,
                                greenlist_dir=GREENLIST_DIR, split=SPLIT)

# module must reproduce the notebook-side stats exactly (sanity_df row order:
# examples x [plain_output, watermarked_output])
module_stats = []
for _, r in examples.iterrows():
    for col in ("plain_output", "watermarked_output"):
        module_stats.append(topic_detection.detect_topic_watermark(
            r[col], tokenizer, state, vocab_size=VOCAB_SIZE))

pairs = zip(module_stats, sanity_df.to_dict("records"))
same = all(m["z_score"] == s["z_score"] and m["confirmed"] == s["confirmed"]
           for m, s in pairs)
print(f"module reproduces notebook detection stats: {same}")
assert same, "exported module disagrees with the notebook"

texts = examples.rename(columns={"watermarked_output": "text"})[["prompt", "topic", "text"]]
topic_detection.detect_dataframe(texts, tokenizer, state, vocab_size=VOCAB_SIZE)

module reproduces notebook detection stats: True


,prompt,topic,text,topic,topic_score,ownership_score,match_count,num_positions,gamma,z_score,p_value,confirmed
0,The government passed a new law before the ele...,history,The government passed a new law before the ele...,history,0.2721,0.1525,9,59,0.11545,0.8916,0.237022,False
1,The cricket team won the final match,sports,The cricket team won the final match of the To...,sports,0.3134,0.2456,14,57,0.13349,2.4889,0.015928,False
2,The new AI chip runs twice as fast,technology,The new AI chip runs twice as fast as today's ...,technology,0.3794,0.4138,24,58,0.22361,3.4760,0.000922,False


In [33]:
# ---- live check: one LONGER prompt per topic --------------------------------
# delta = 2.0 (generation pipeline's operating point), long generations so the
# z-statistic has enough scored tokens to cross the threshold.
DELTA = 2.0
torch.manual_seed(0)

LONG_PROMPTS_PER_TOPIC = {
    "technology": ("The new AI chip announced this week runs twice as fast as the previous "
                   "generation while drawing far less power, and analysts say its improved memory "
                   "bandwidth could reshape the entire market for data center hardware."),
    "medicine": ("Doctors tested the new vaccine across three hospitals last winter, enrolling "
                 "thousands of volunteers in a randomized trial that carefully tracked side effects "
                 "and antibody levels for six months after the final dose."),
    "sports": ("The cricket team chased down 280 runs in the final match and won the trophy on the "
               "very last ball, capping a tournament full of close finishes and record-breaking "
               "batting performances from younger players."),
    "politics": ("The government passed a controversial new law just before the election, and the "
                 "opposition immediately challenged it in court, arguing that the rushed midnight "
                 "vote ignored decades of constitutional precedent."),
    "science": ("Scientists discovered an unexpected new particle at the collider last spring, and "
                "independent teams have spent months reanalyzing the collision data to confirm "
                "whether the signal survives every statistical check."),
    "entertainment": ("The movie sequel broke box office records opening weekend, selling out "
                      "late-night screenings in dozens of cities and earning praise for its "
                      "practical effects despite mixed reviews from longtime fans."),
    "finance": ("Central bank rates pushed markets down again this quarter, and investors rotated "
                "out of growth stocks into bonds as inflation reports kept coming in hotter than "
                "most economists had forecast earlier in the year."),
    "history": ("Ancient Rome fell after centuries of slow decline, as repeated invasions, economic "
                "collapse and constant political instability gradually dismantled an empire that "
                "had once governed the entire Mediterranean world."),
}

rows = []
for expected, prompt in LONG_PROMPTS_PER_TOPIC.items():
    topic, _ = extract_topic(prompt)                     # route exactly like generation
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    processor = TopicBoostProcessor(greenlists[topic][SPLIT], delta=DELTA)

    with torch.no_grad():
        plain = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                               temperature=1.0, top_p=0.9)
        boosted = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                 temperature=1.0, top_p=0.9,
                                 logits_processor=[processor])

    det_plain = detect_topic_watermark(tokenizer.decode(plain[0], skip_special_tokens=True))
    det_wm    = detect_topic_watermark(tokenizer.decode(boosted[0], skip_special_tokens=True))

    rows.append({
        "expected_topic": expected,
        "routed_topic": topic,
        "prompt_toks": len(tokenizer.encode(prompt, add_special_tokens=False)),
        "n": det_wm["num_positions"],
        "plain_z": det_plain["z_score"], "plain_ok": det_plain["confirmed"],
        "wm_hits": det_wm["match_count"], "wm_gamma": det_wm["gamma"],
        "wm_rate": det_wm["ownership_score"], "wm_z": det_wm["z_score"],
        "wm_p": det_wm["p_value"], "wm_ok": det_wm["confirmed"],
    })
    print(f"{expected:13} z: {det_plain['z_score']:+6.2f} -> {det_wm['z_score']:+6.2f}"
          f"  confirmed: {det_plain['confirmed']} -> {det_wm['confirmed']}")

long_df = pd.DataFrame(rows)
routing_ok = long_df["routed_topic"] == long_df["expected_topic"]
print(f"\nrouting accuracy: {routing_ok.mean():.0%}")
print(f"mean z: plain = {long_df.plain_z.mean():+.2f} | watermarked = {long_df.wm_z.mean():+.2f}")
long_df

technology    z:  -2.40 ->  +8.94  confirmed: False -> True
medicine      z:  +0.56 ->  +6.26  confirmed: False -> True
sports        z:  -2.75 ->  +6.96  confirmed: False -> True
politics      z:  -1.21 -> +14.78  confirmed: False -> True
science       z:  -0.84 -> +10.79  confirmed: False -> True
entertainment z:  +0.41 ->  +9.20  confirmed: False -> True
finance       z:  +0.38 ->  +7.28  confirmed: False -> True
history       z:  -0.07 -> +15.40  confirmed: False -> True

routing accuracy: 25%
mean z: plain = -0.74 | watermarked = +9.95


,expected_topic,routed_topic,prompt_toks,n,plain_z,plain_ok,wm_hits,wm_gamma,wm_rate,wm_z,wm_p,wm_ok
0,technology,technology,39,239,-2.3976,False,111,0.22361,0.4644,8.9352,1.962988e-16,True
1,medicine,history,36,236,0.5610,False,58,0.11545,0.2458,6.2646,1.914952e-08,True
2,sports,history,39,239,-2.7513,False,62,0.11545,0.2594,6.9647,6.611676e-10,True
3,politics,history,33,233,-1.2094,False,99,0.11545,0.4249,14.7811,6.872593e-33,True
4,science,history,35,235,-0.8431,False,80,0.11545,0.3404,10.7924,1.118211e-19,True
5,entertainment,history,34,234,0.4061,False,72,0.11545,0.3077,9.2025,3.067921e-15,True
6,finance,history,36,236,0.3817,False,63,0.11545,0.2669,7.2831,1.363731e-10,True
7,history,history,33,233,-0.0706,False,102,0.11545,0.4378,15.3961,3.452266e-35,True
